<a href="https://colab.research.google.com/github/asiahcrutchfield/ground-level-languages-ai/blob/ai-audio/gll_intro_to_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==============================================================================
# Ground Level Languages — Accurate Phoneme Extraction (Wav2Vec2-eSpeak + MMS_FA)
# ==============================================================================

!pip install -q transformers torch torchaudio torchcrepe librosa soundfile scipy numpy

import os
import json
import zipfile
import unicodedata
import numpy as np
import torch
import torchaudio
import torchcrepe
import librosa
import soundfile as sf
from scipy.signal import find_peaks
from huggingface_hub import hf_hub_download
from transformers import Wav2Vec2ForCTC, Wav2Vec2FeatureExtractor

# 1. Setup output directories & device
os.makedirs("audio/slices", exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Load Wav2Vec2-eSpeak directly (bypasses phonemizer/espeak-ng ImportError)
MODEL_ID = "facebook/wav2vec2-lv-60-espeak-cv-ft"
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)
phoneme_model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID).to(device).eval()

vocab_file = hf_hub_download(repo_id=MODEL_ID, filename="vocab.json")
with open(vocab_file, "r", encoding="utf-8") as f:
    vocab_dict = json.load(f)
id_to_token = {v: k for k, v in vocab_dict.items()}
special_tokens = {"<pad>", "<s>", "</s>", "<unk>", "|"}

def extract_ipa_phonemes(audio_16k):
    inputs = feature_extractor(audio_16k, sampling_rate=16000, return_tensors="pt")
    with torch.inference_mode():
        logits = phoneme_model(inputs.input_values.to(device)).logits
    pred_ids = torch.argmax(logits, dim=-1)[0].cpu().tolist()

    # CTC greedy collapse (deduplicate consecutive IDs & drop <pad> / special tokens)
    phones = []
    prev_id = None
    for tid in pred_ids:
        if tid != prev_id:
            tok = id_to_token.get(tid, "")
            if tok and tok not in special_tokens:
                phones.append(tok)
        prev_id = tid
    return phones

# 3. Load MMS_FA Forced Aligner for exact millisecond phoneme boundaries
bundle = torchaudio.pipelines.MMS_FA
fa_model = bundle.get_model().to(device)
fa_tokenizer = bundle.get_tokenizer()
fa_aligner = bundle.get_aligner()

IPA_BASE_VOWELS = set("aeiouyɨʉɯuɪʏʊeøɘɵɤoəɛœɜɞʌɔæɐaɶɑɒɚɝ")

IPA_TO_MMS_CHAR = {
    "k": "k", "t": "t", "p": "p", "d": "d", "ɡ": "g", "g": "g", "b": "b",
    "dʒ": "j", "tʃ": "c", "w": "w", "l": "l", "r": "r", "ɹ": "r",
    "m": "m", "n": "n", "ŋ": "n", "s": "s", "z": "z", "ʃ": "s",
    "f": "f", "v": "v", "h": "h", "θ": "t", "ð": "d",
    "æ": "a", "ɑ": "a", "a": "a", "ɔ": "o", "o": "o", "ʌ": "u", "ə": "e",
    "ɛ": "e", "e": "e", "ɪ": "i", "i": "i", "ʊ": "u", "u": "u", "ɚ": "r"
}

def normalize_ipa(raw_phone):
    stripped = raw_phone.replace("ʰ", "").replace("ː", "").replace("ˈ", "").replace("ˌ", "")
    decomposed = unicodedata.normalize("NFD", stripped)
    return "".join(ch for ch in decomposed if unicodedata.category(ch) != "Mn")

def classify_phone(raw_phone):
    clean = normalize_ipa(raw_phone)
    return "vowel" if any(ch in IPA_BASE_VOWELS for ch in clean) else "consonant"

def ipa_to_fa_tokens(ipa_list):
    chars = []
    for ipa in ipa_list:
        clean = normalize_ipa(ipa)
        chars.append(IPA_TO_MMS_CHAR.get(clean, IPA_TO_MMS_CHAR.get(clean[:1], "a")))
    return "".join(chars)

def apply_fade(clip, sr=16000, fade_ms=10):
    fade_len = min(int(sr * (fade_ms / 1000.0)), len(clip) // 2)
    if fade_len <= 1:
        return clip
    win = np.ones(len(clip), dtype=np.float32)
    win[:fade_len] = np.linspace(0.0, 1.0, fade_len)
    win[-fade_len:] = np.linspace(1.0, 0.0, fade_len)
    return clip * win

def synthesize_humming_track(y, f0, periodicity, sr=16000, hop_length=160):
    total_samples = len(y)
    freq_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(f0)), f0)
    conf_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(periodicity)), periodicity)
    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=hop_length)[0]
    rms_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(rms)), rms)
    rms_norm = rms_interp / (np.max(rms_interp) + 1e-6)
    voiced_mask = (conf_interp > 0.15) & (rms_norm > 0.12)
    amp = np.where(voiced_mask, 0.35 * rms_norm, 0.0)
    phase = (2.0 * np.pi * np.cumsum(freq_interp)) / sr
    return (amp * (0.8 * np.sin(phase) + 0.2 * np.sin(2.0 * phase))).astype(np.float32)

def process_word(word_id, mp3_path, visual_path):
    y_raw, sr = librosa.load(mp3_path, sr=16000)
    y, _ = librosa.effects.trim(y_raw, top_db=28)
    y = np.pad(y, (int(0.05 * sr), int(0.05 * sr)), mode="constant")

    # STEP 1: Extract clean IPA phonemes with Wav2Vec2-eSpeak
    detected_phones = extract_ipa_phonemes(y)
    print(f"[{word_id}] Wav2Vec2-eSpeak extracted phonemes -> {detected_phones}")

    # Align extracted phonemes to millisecond timestamps with MMS_FA
    fa_transcript = ipa_to_fa_tokens(detected_phones) if detected_phones else word_id
    waveform = torch.tensor(y).unsqueeze(0).to(device)
    with torch.inference_mode():
        emission, _ = fa_model(waveform)
    tokens = fa_tokenizer([fa_transcript])
    token_spans = fa_aligner(emission[0], tokens)[0]
    ratio = waveform.size(1) / emission.size(1)

    phonemes_out = []
    for idx, (span, ipa) in enumerate(zip(token_spans, detected_phones)):
        s_idx = max(0, int(span.start * ratio) - int(0.015 * sr))
        e_idx = min(len(y), int(span.end * ratio) + int(0.020 * sr))
        clip = apply_fade(y[s_idx:e_idx], sr=sr, fade_ms=10)
        clip_path = f"audio/slices/{word_id}_p{idx+1}.wav"
        sf.write(clip_path, clip, sr)
        phonemes_out.append({
            "id": f"{word_id}_p{idx+1}",
            "kind": classify_phone(ipa),  # "consonant" or "vowel" (zero IPA leaked to UI)
            "audio_url": clip_path
        })

    # STEP 2: Syllable duration segmentation
    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=160)[0]
    peaks, _ = find_peaks(rms, distance=12, prominence=np.max(rms) * 0.2)
    if len(peaks) == 0:
        peaks = [len(rms) // 2]
    boundaries = [0] + [int((peaks[i] + peaks[i+1]) / 2) for i in range(len(peaks) - 1)] + [len(rms)]
    syllables_out = []
    for idx in range(len(boundaries) - 1):
        s_s, e_s = (boundaries[idx] * 160) / sr, (boundaries[idx+1] * 160) / sr
        dur = max(0.08, e_s - s_s)
        clip_path = f"audio/slices/{word_id}_s{idx+1}.wav"
        sf.write(clip_path, apply_fade(y[int(s_s * sr):int(e_s * sr)], sr=sr), sr)
        syllables_out.append({
            "id": f"{word_id}_s{idx+1}",
            "width_px": int(min(180, max(65, dur * 240))),
            "audio_url": clip_path
        })

    # STEP 3: TorchCREPE F0 Pitch Contour & Stripped Humming Track
    pitch, periodicity = torchcrepe.predict(
        torch.tensor(y).unsqueeze(0), sr, hop_length=160, fmin=65.0, fmax=500.0,
        model="full", decoder=torchcrepe.decode.viterbi, return_periodicity=True, device=device
    )
    frequency = pitch.squeeze(0).cpu().numpy()
    conf = periodicity.squeeze(0).cpu().numpy()
    hum_out = f"audio/slices/{word_id}_hum.wav"
    sf.write(hum_out, synthesize_humming_track(y, frequency, conf, sr=sr), sr)

    tones_out = []
    for idx in range(len(syllables_out)):
        s_f = int((idx / len(syllables_out)) * len(frequency))
        e_f = int(((idx + 1) / len(syllables_out)) * len(frequency))
        slope = float(frequency[e_f-1] - frequency[s_f]) if e_f - s_f > 1 else 0.0
        style = "solid" if slope > 12 else ("ring" if slope < -12 else "dot")
        tones_out.append({"id": f"{word_id}_t{idx+1}", "style": style, "audio_url": hum_out})

    return {
        "id": word_id,
        "visual_url": visual_path,
        "full_audio_url": f"audio/{word_id}.mp3",
        "humming_audio_url": hum_out,
        "phonemes": phonemes_out,
        "syllables": syllables_out,
        "tones": tones_out
    }

# 4. Run on your 5 Level 0 words
words = [
    ("cat", "/content/cat.mp3", "images/visuals/cat.jpg"),
    ("dog", "/content/dog.mp3", "images/visuals/dog.jpg"),
    ("walk", "/content/walk.mp3", "images/visuals/walk.gif"),
    ("jump", "/content/jump.mp3", "images/visuals/jump.gif"),
    ("tree", "/content/tree.mp3", "images/visuals/tree.jpg")
]

dataset = {"items": [process_word(w, p, v) for w, p, v in words if os.path.exists(p)]}
with open("lesson_data.json", "w") as f:
    json.dump(dataset, f, indent=2)

# 5. Zip lesson_data.json + audio/slices/ for your repo
with zipfile.ZipFile("gll_update2_assets.zip", "w") as zf:
    zf.write("lesson_data.json")
    for root, _, files in os.walk("audio/slices"):
        for file in files:
            zf.write(os.path.join(root, file))

print("\nDone! Extract gll_update2_assets.zip into your project root.")

preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Downloading: "https://dl.fbaipublicfiles.com/mms/torchaudio/ctc_alignment_mling_uroman/model.pt" to /root/.cache/torch/hub/checkpoints/model.pt


100%|██████████| 1.18G/1.18G [00:09<00:00, 139MB/s]



Done! Extract gll_update2_assets.zip into your project root.
